# Permian Basin Subset

**Pipeline stage:** Step 3 of the Texas RRC oil production data pipeline (follows the notebook that
produces well coordinates and well-level production/disposition estimates).

This notebook takes the statewide, Texas-wide outputs from the previous pipeline stage and filters
them down to just the **Permian Basin**, saving lightweight CSV extracts for downstream spatial
analysis (e.g. matching wells to satellite-detected flaring). It produces no new data — it's a pure
geographic subset plus a column trim, so the outputs stay small and easy to work with.

Three Permian-only files are produced:

| Output file | Grain | Source |
|---|---|---|
| `permian_wells_with_locn_and_id.csv` | one row per well | well coordinates |
| `permian_wells_per_lease.csv` | one row per lease | well-count-per-lease |
| `permian_prod_per_well_approx.csv` | one row per well per month | well-level production/disposition estimates |

Only columns that are actually useful downstream are kept in each file — see Section 0 for exactly
which columns are dropped and why.

## 0. Overview: What Gets Filtered, and How

### The Permian Basin bounding box

All three outputs are filtered using the same rectangular bounding box (in WGS84 decimal degrees),
covering the Permian Basin play area across West Texas and southeastern New Mexico:

| Bound | Value |
|---|---|
| Latitude min | 29.462935° N |
| Latitude max | 34.021515° N |
| Longitude min | −105.21988° W |
| Longitude max | −100.036107° W |

A row is kept if its coordinate falls inside this box. This is a simple rectangular filter, not a
precise basin-boundary polygon — some rows near the edges of the actual geologic Permian Basin
boundary may be included or excluded depending on how closely the true boundary hugs this rectangle.

### Filtering a per-lease file without a lease-level coordinate

The previous pipeline stage no longer computes a single averaged (centroid) coordinate per lease —
every well keeps its own individual coordinate instead (see that notebook's Section 0 for why). That
means `wells_per_lease.parquet` (the well-count-per-lease file) has **no coordinate columns to filter
on at all** — it's just `lease_key` + `n_wells_with_coordinates`.

To still produce a Permian-only version of that file, this notebook determines **which leases have
at least one well inside the bounding box** using the well-level coordinate file (which does have
coordinates), and then keeps only the wells-per-lease rows for those lease keys. This is a small
change in mechanism from geographic bounding-box filtering to a lease-key membership filter, but the
practical effect — "leases situated in the Permian" — is the same.

### Reading only the columns each output needs

Each source file carries a number of columns that exist for pipeline bookkeeping (raw RRC key
components, shapefile provenance metadata, intermediate normalization columns) but aren't useful in
a final, analysis-ready CSV. Rather than reading every column and dropping the unneeded ones
afterward, this notebook asks `pandas`/`geopandas` to read **only the columns each output actually
needs** directly off disk — cheaper on memory, and it makes explicit exactly what's being kept.

For example, `lease_well_coordinates.geoparquet` carries raw/intermediate columns like
`oil_gas_code`, `api_county_code`, `api8_from_components`, `SOURCE_ZIP`, `SOURCE_SHP`,
`well_layer_type`, and (when present in the source shapefiles) `API`, `API10`, `APINUM`, `LAT27`,
`LONG27`, `LAT83`, `LONG83`, `RELIAB`, `SYMNUM`, `WELLID` — all of these are either superseded by a
cleaner derived column (`api8` supersedes the raw API component/candidate columns) or are internal
shapefile bookkeeping with no analytical use once coordinates have been assigned. None of them are
read in Section 2 below.

## 1. Setup

In [1]:
# ── Imports + logging ─────────────────────────────────────────────────────────

from pathlib import Path
import logging
import sys

import pandas as pd
import geopandas as gpd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger(__name__)

log.info("✓ Imports + logging ready.")


09:54:50 [INFO] ✓ Imports + logging ready.


## 2. Configuration

- `well_loc` — `lease_well_coordinates.geoparquet`: one row per well, with coordinates (from the
  previous pipeline stage).
- `wells_per_lease_path` — `wells_per_lease.parquet`: one row per lease, with
  `n_wells_with_coordinates` (no coordinates — see Section 0).
- `prod_disp_path` — `prod_per_well_approx.geoparquet`: one row per well per month, with
  disposition volumes already equally split across wells and its own longitude/latitude.
- `output_folder` — a `permian_only` subfolder of the shared cleaned-data directory, so statewide and
  Permian-only outputs are never mixed up in the same folder.
- The four bounding-box constants described in Section 0.

In [2]:
well_loc            = Path("../../../../data/processed/texas/lease_well_coordinates.geoparquet")
wells_per_lease_path = Path("../../../../data/processed/texas/wells_per_lease.parquet")
prod_disp_path      = Path("../../../../data/processed/texas/prod_per_well_approx.geoparquet")

output_folder = Path("../../../../data/processed/texas/permian_only")
output_folder.mkdir(parents=True, exist_ok=True)

# Permian Basin bounding box (WGS84 decimal degrees) — see Section 0
lat_min, lat_max = 29.462935, 34.021515
long_min, long_max = -105.21988, -100.036107

log.info("✓ Config set. Outputs will be written to: %s", output_folder)


09:54:50 [INFO] ✓ Config set. Outputs will be written to: ../../../../data/processed/texas/permian_only


## 3. Part A — Individual Wells With Coordinates and ID

Reads only the columns needed for a clean, well-level "where is this well and what is it" reference
file: `lease_key` (which lease it belongs to), `api8` (its unique RRC well identifier),
`well_no`, `county_name`, `wellbore_location_code` (land/offshore/inland waterway/bay), and its
coordinates. Everything else on the source file (raw key components, shapefile provenance, optional
raw API/lat-long variants) is left unread.

In [3]:
well_cols = ["lease_key", "api8", "well_no", "county_name", "wellbore_location_code",
             "longitude", "latitude", "geometry"]

texas_wells = gpd.read_parquet(well_loc, columns=well_cols)

print("Texas wells (all counties):", texas_wells.shape)
texas_wells.head()


Texas wells (all counties): (587614, 8)


,lease_key,api8,well_no,county_name,wellbore_location_code,longitude,latitude,geometry
0,O_08_00277,04931721,17,BROWN,L,-99.098679,32.045194,POINT (-99.09868 32.04519)
1,O_08_00277,04932950,18,BROWN,L,-99.101261,32.045274,POINT (-99.10126 32.04527)
2,O_08_00287,04905559,1,BROWN,L,-99.085963,32.056174,POINT (-99.08596 32.05617)
3,O_08_00291,04905919,1,BROWN,L,-99.059271,32.039860,POINT (-99.05927 32.03986)
4,O_08_00291,04980560,2,BROWN,L,-99.059242,32.041674,POINT (-99.05924 32.04167)


In [4]:
texas_permian_wells = texas_wells.loc[
    (texas_wells["latitude"] >= lat_min) &
    (texas_wells["latitude"] <= lat_max) &
    (texas_wells["longitude"] >= long_min) &
    (texas_wells["longitude"] <= long_max)
].copy()

del texas_wells

print("Permian wells:", texas_permian_wells.shape)
texas_permian_wells.head()


Permian wells: (295418, 8)


,lease_key,api8,well_no,county_name,wellbore_location_code,longitude,latitude,geometry
1389,O_10_00875,13502096,67W,ECTOR,L,-102.422757,31.791135,POINT (-102.42276 31.79113)
1390,O_10_00875,13510238,68,ECTOR,L,-102.426877,31.790239,POINT (-102.42688 31.79024)
1391,O_10_00875,13510238,68W,ECTOR,L,-102.426877,31.790239,POINT (-102.42688 31.79024)
1392,O_10_00875,13502111,71,ECTOR,L,-102.440639,31.785008,POINT (-102.44064 31.78501)
1393,O_10_00875,13502110,72,ECTOR,L,-102.436435,31.786058,POINT (-102.43643 31.78606)


A quick visual check — a random sample of the filtered wells plotted on an interactive map should
show a cluster confined to West Texas / southeastern New Mexico, not scattered across the whole
state.

In [5]:
sample_n = min(5000, texas_permian_wells["geometry"].notna().sum())

texas_permian_wells.dropna(subset=["geometry"]).sample(
    sample_n,
    random_state=42
).explore(
    tiles="CartoDB positron",
    tooltip=["lease_key", "api8"],
)


Save as CSV. Since CSV can't store native geometry objects, the point geometry is also written out
as a `geometry_wkt` (Well-Known Text) string column.

In [6]:
permian_lease_keys = texas_permian_wells["lease_key"].unique()
print(f"Permian leases represented by these wells: {len(permian_lease_keys):,}")

texas_permian_wells["geometry_wkt"] = texas_permian_wells.geometry.to_wkt()
texas_permian_wells.drop(columns="geometry").to_csv(
    output_folder / "permian_wells_with_locn_and_id.csv", index=False
)

log.info("✓ Saved permian_wells_with_locn_and_id.csv (%s rows)", f"{len(texas_permian_wells):,}")

del texas_permian_wells


Permian leases represented by these wells: 72,567
09:54:53 [INFO] ✓ Saved permian_wells_with_locn_and_id.csv (295,418 rows)


## 4. Part B — Well Counts per Permian Lease

`wells_per_lease.parquet` has no coordinates to filter on (Section 0), so this uses the
`permian_lease_keys` computed in Part A — every lease that has at least one well inside the bounding
box — to subset it by `lease_key` membership instead. The file is already minimal
(`lease_key` + `n_wells_with_coordinates`), so there are no further columns to trim.

In [7]:
wells_per_lease = pd.read_parquet(wells_per_lease_path)
print("Wells-per-lease, all of Texas:", wells_per_lease.shape)

permian_wells_per_lease = wells_per_lease[
    wells_per_lease["lease_key"].isin(permian_lease_keys)
].copy()

del wells_per_lease

print("Wells-per-lease, Permian only:", permian_wells_per_lease.shape)
permian_wells_per_lease.head()


Wells-per-lease, all of Texas: (169051, 2)
Wells-per-lease, Permian only: (72567, 2)


,lease_key,n_wells_with_coordinates
943,O_01_02545,10
960,O_01_02592,17
1145,O_01_02969,20
1382,O_01_03407,9
1383,O_01_03408,1


In [8]:
permian_wells_per_lease.to_csv(output_folder / "permian_wells_per_lease.csv", index=False)
log.info("✓ Saved permian_wells_per_lease.csv (%s rows)", f"{len(permian_wells_per_lease):,}")

del permian_wells_per_lease


09:54:53 [INFO] ✓ Saved permian_wells_per_lease.csv (72,567 rows)


## 5. Part C — Well-Level Production/Disposition Estimates

`prod_per_well_approx.geoparquet` carries its own longitude/latitude per row (every row is already
one well, one month), so it's filtered directly by coordinate, the same way as Part A. All of its
columns are relevant for the full output — nothing is dropped here yet.

In [9]:
prod_per_well = gpd.read_parquet(prod_disp_path)

print("Well-level production, all of Texas:", prod_per_well.shape)
print("Date range:", prod_per_well["date"].min(), "→", prod_per_well["date"].max())


Well-level production, all of Texas: (51189274, 34)
Date range: 2012-02-01 00:00:00 → 2026-03-01 00:00:00


In [10]:
permian_prod_disp = prod_per_well.loc[
    (prod_per_well["latitude"] >= lat_min) &
    (prod_per_well["latitude"] <= lat_max) &
    (prod_per_well["longitude"] >= long_min) &
    (prod_per_well["longitude"] <= long_max)
].copy()

del prod_per_well

print("Well-level production, Permian only:", permian_prod_disp.shape)
print(f"Unique Permian leases represented: {permian_prod_disp['lease_key'].nunique():,}")
permian_prod_disp.head()


Well-level production, Permian only: (32014046, 34)
Unique Permian leases represented: 47,919


,field_no,operator_no,operator_name,oil_pipeline_bbl,oil_truck_bbl,oil_tankcar_bbl,oil_tank_cleaning_bbl,oil_circulating_bbl,oil_lost_stolen_bbl,oil_bsw_repressure_bbl,...,total_vented_flared_mcf,date,lease_key,api8,well_no,county_name,longitude,latitude,geometry,n_wells_with_coordinates
110,14321500,885555,VICTORIAN OIL & GAS,0.0,161.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2012-04-01,O_10_22061,47530372,1,WARD,-102.923316,31.596287,POINT (-102.92332 31.59629),1.0
111,14321500,885555,VICTORIAN OIL & GAS,0.0,140.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2012-09-01,O_10_22061,47530372,1,WARD,-102.923316,31.596287,POINT (-102.92332 31.59629),1.0
112,14321500,885555,VICTORIAN OIL & GAS,0.0,150.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2012-11-01,O_10_22061,47530372,1,WARD,-102.923316,31.596287,POINT (-102.92332 31.59629),1.0
113,14092500,829482,SUNDOWN ENERGY LP,0.0,30.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2012-02-01,O_10_25715,22732018,1,HOWARD,-101.625771,32.433690,POINT (-101.62577 32.43369),2.0
114,14092500,829482,SUNDOWN ENERGY LP,0.0,30.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2012-02-01,O_10_25715,22732262,2,HOWARD,-101.621242,32.430529,POINT (-101.62124 32.43053),2.0


### 5.1 Save the Full Version

Same pattern as Part A: geometry is converted to WKT for CSV storage. Drops `field_no`, `operator_name`,
`well_no`, `county_name`, and the (already-converted-to-WKT) `geometry` column. `api8`, `lease_key`,
`operator_no`, `longitude`/`latitude`, `n_wells_with_coordinates`, and every production/disposition
column are kept.

In [11]:
permian_prod_disp["geometry_wkt"] = permian_prod_disp.geometry.to_wkt()

permian_prod_disp = permian_prod_disp.drop(columns=[
    "field_no", "operator_name", "well_no", "county_name", "geometry",
])

permian_prod_disp.to_csv(
    output_folder / "permian_prod_per_well_approx.csv", index=False
)

log.info("✓ Saved permian_prod_per_well_approx.csv (%s rows, %s columns)",
         f"{len(permian_prod_disp):,}", permian_prod_disp.shape[1])



del permian_prod_disp

10:01:05 [INFO] ✓ Saved permian_prod_per_well_approx.csv (32,014,046 rows, 30 columns)


## 6. Summary

This notebook produced four files in `output_folder`:

| File | Grain | Description |
|---|---|---|
| `permian_wells_with_locn_and_id.csv` | one row per well | `lease_key`, `api8`, `well_no`, `county_name`, `wellbore_location_code`, coordinates |
| `permian_wells_per_lease.csv` | one row per lease | `lease_key`, `n_wells_with_coordinates` |
| `permian_prod_per_well_approx.csv` | one row per well per month | Full well-level production/disposition estimate, Permian only |
